# 🔐 Post-Quantum Cryptography with liboqs
### Exploring Quantum-Safe algorithms using Open Quantum Safe (OQS) library
- **Library**: liboqs-python
- **Purpose**: Learn and test Post-Quantum Cryptographic algorithms
- **Date**: June 2025

In [1]:
#Import and verify liboqs

import oqs
import os
import hashlib
import json
import cryptography
import struct

print("✅ liboqs imported successfully!")

print(f"📦 Total KEM algorithms available : {len(oqs.get_enabled_kem_mechanisms())}")
print(f"📦 Total SIG algorithms available : {len(oqs.get_enabled_sig_mechanisms())}")
print(f"🔐 liboqs is ready to use!")



liboqs-python faulthandler is disabled
✅ liboqs imported successfully!
📦 Total KEM algorithms available : 29
📦 Total SIG algorithms available : 221
🔐 liboqs is ready to use!


In [2]:
#List all available algorithms

kem_list = oqs.get_enabled_kem_mechanisms()
sig_list = oqs.get_enabled_sig_mechanisms()

print("Total KEM Algorithms: ", len(kem_list))
print("Total SIG Algorithms: ", len(sig_list))

print("\n KEM Algorithms: ")
for i, kem in enumerate(kem_list,1):
    print(f" {i:02}.{kem}")

print("\n Signature Algorithms: ")
for i, sig in enumerate(sig_list,1):
    print(f" {i:02}.{sig}")

Total KEM Algorithms:  29
Total SIG Algorithms:  221

 KEM Algorithms: 
 01.Classic-McEliece-348864
 02.Classic-McEliece-348864f
 03.Classic-McEliece-460896
 04.Classic-McEliece-460896f
 05.Classic-McEliece-6688128
 06.Classic-McEliece-6688128f
 07.Classic-McEliece-6960119
 08.Classic-McEliece-6960119f
 09.Classic-McEliece-8192128
 10.Classic-McEliece-8192128f
 11.Kyber512
 12.Kyber768
 13.Kyber1024
 14.ML-KEM-512
 15.ML-KEM-768
 16.ML-KEM-1024
 17.NTRU-HPS-2048-509
 18.NTRU-HPS-2048-677
 19.NTRU-HPS-4096-821
 20.NTRU-HPS-4096-1229
 21.NTRU-HRSS-701
 22.NTRU-HRSS-1373
 23.sntrup761
 24.FrodoKEM-640-AES
 25.FrodoKEM-640-SHAKE
 26.FrodoKEM-976-AES
 27.FrodoKEM-976-SHAKE
 28.FrodoKEM-1344-AES
 29.FrodoKEM-1344-SHAKE

 Signature Algorithms: 
 01.ML-DSA-44
 02.ML-DSA-65
 03.ML-DSA-87
 04.Falcon-512
 05.Falcon-1024
 06.Falcon-padded-512
 07.Falcon-padded-1024
 08.SPHINCS+-SHA2-128f-simple
 09.SPHINCS+-SHA2-128s-simple
 10.SPHINCS+-SHA2-192f-simple
 11.SPHINCS+-SHA2-192s-simple
 12.SPHINCS+-S

In [3]:
#Explore details of each KEM Algorithm

print(f"{'Algorithm':<35} {'PubKey':>8} {'SecKey':>8} {'Cipher':>8} {'Secret':>8}")
print('-'*80)

for algo in oqs.get_enabled_kem_mechanisms():
    with oqs.KeyEncapsulation(algo) as kem:
        d=kem.details
        print(f"{d['name']:<35} "
              f"{d['length_public_key']:>8} "
              f"{d['length_secret_key']:>8} "
              f"{d['length_ciphertext']:>8} "
              f"{d['length_shared_secret']:>8} "
              f"{d['claimed_nist_level']:>6}")

Algorithm                             PubKey   SecKey   Cipher   Secret
--------------------------------------------------------------------------------
Classic-McEliece-348864               261120     6492       96       32      1
Classic-McEliece-348864f              261120     6492       96       32      1
Classic-McEliece-460896               524160    13608      156       32      3
Classic-McEliece-460896f              524160    13608      156       32      3
Classic-McEliece-6688128             1044992    13932      208       32      5
Classic-McEliece-6688128f            1044992    13932      208       32      5
Classic-McEliece-6960119             1047319    13948      194       32      5
Classic-McEliece-6960119f            1047319    13948      194       32      5
Classic-McEliece-8192128             1357824    14120      208       32      5
Classic-McEliece-8192128f            1357824    14120      208       32      5
Kyber512                                 800     1632    

In [4]:
# Kem - Key Exchange Demo

algorithms = ['ML-KEM-512', 'ML-KEM-768', 'ML-KEM-1024']

for algo in algorithms:
    with oqs.KeyEncapsulation(algo) as kem:
        #Generate Key pair
        public_key = kem.generate_keypair()

        #Encapsulate
        ciphertext, shared_secret_sender = kem.encap_secret(public_key)

        #Decapsulate
        shared_secret_receiver = kem.decap_secret(ciphertext)

        match = shared_secret_sender == shared_secret_receiver

        print(f"   Algorithm     : {algo}")
        print(f"   Public Key    : {len(public_key)} bytes")
        print(f"   Ciphertext    : {len(ciphertext)} bytes")
        print(f"   Shared Secret : {len(shared_secret_sender)} bytes")
        print(f"   Secrets Match : {'YES' if match else 'NO'}")
        print()

   Algorithm     : ML-KEM-512
   Public Key    : 800 bytes
   Ciphertext    : 768 bytes
   Shared Secret : 32 bytes
   Secrets Match : YES

   Algorithm     : ML-KEM-768
   Public Key    : 1184 bytes
   Ciphertext    : 1088 bytes
   Shared Secret : 32 bytes
   Secrets Match : YES

   Algorithm     : ML-KEM-1024
   Public Key    : 1568 bytes
   Ciphertext    : 1568 bytes
   Shared Secret : 32 bytes
   Secrets Match : YES



In [5]:
#Digital Signature - Sign and Verify Demo

sig_algorithms = ['ML-DSA-44', 'ML-DSA-65', 'ML-DSA-87']

keygen_times = []
sign_times = []
verify_times = []
ITERATIONS = 1000

message = b"Hello Post Quantum World! This message is quantum safe-signed!"

for algo in sig_algorithms:
    try:
        for _ in range(ITERATIONS):
            with oqs.Signature(algo) as signer:
                #Generate Key Pair
                t0 = time.perf_counter()
                public_key = signer.generate_keypair()
                keygen_times.append(time.perf_counter() - t0)
                
    
                #Sign
                t0 = time.perf_counter()
                signature = signer.sign(message)
                sign_times.append(time.perf_counter() - t0)
    
                #Verify
                t0 = time.perf_counter()
                is_valid = signer.verify(message, signature, public_key)
                verify_times.append(time.perf_counter() - t0)
    
                avg_kg = sum(keygen_times) / ITERATIONS * 1000
                avg_en = sum(sign_times)  / ITERATIONS * 1000
                avg_de = sum(verify_times)  / ITERATIONS * 1000
                total  = avg_kg + avg_en + avg_de

        print(f"   Algorithm   : {algo}")
        print(f"   Public Key  : {len(public_key)} bytes")
        print(f"   Signature   : {len(signature)} bytes")
        print(f"   Valid       : {'YES' if is_valid else 'NO'}")
        print()
        print(f"{'Algorithm':<25} {'KeyGen':>10} {'Signing':>10} {'Verification':>10} {'Total':>10}")
        print("-" * 65)
        print(f"{algo:<25} {avg_kg:>9.3f}ms {avg_en:>9.3f}ms {avg_de:>9.3f}ms {total:>9.3f}ms")
        print()
        print()
            
    except Exception as e:
        print(f"  {algo} not available: {e}\n")

  ML-DSA-44 not available: name 'time' is not defined

  ML-DSA-65 not available: name 'time' is not defined

  ML-DSA-87 not available: name 'time' is not defined



In [6]:
#Tamper Detection Test

with oqs.Signature("ML-DSA-65") as signer:
    public_key = signer.generate_keypair()

    original_message = b"Original Secure Message!!!"
    tampered_message = b"Tampered Secure Message!!!"

    #Sign original
    signature = signer.sign(original_message)

    #Verify Original
    valid_original = signer.verify(original_message, signature, public_key)

    #Verify Tampered
    valid_tampered = signer.verify(tampered_message, signature, public_key)

    print("   Tamper Detection Test")
    print(f"   Original Message Verified : {'PASS' if valid_original else 'FAIL'}")
    print(f"   Tampered Message Verified : {'PASS' if valid_tampered else 'FAIL'}")
    print()
    print("Tamper was detected!" if not valid_tampered else "Tamper NOT detected!")

   Tamper Detection Test
   Original Message Verified : PASS
   Tampered Message Verified : FAIL

Tamper was detected!


In [7]:
#Random Bytes Generator

import oqs.rand as oqsrand

print("Quantum-Safe Random Byte Generation\n")

for size in [16, 32, 64]:
    random_bytes = oqsrand.randombytes(size)
    print(f"   {size} bytes : {random_bytes.hex()}")

Quantum-Safe Random Byte Generation

   16 bytes : c4466706ef54817bcf910aa529e838da
   32 bytes : 1652b3ca44ffc39ec91ac591d0c36f8949a45531ee1519372d9130e8e71e96b9
   64 bytes : 426041ec33092e7c4234a298cde4702ab7694e5894a4acb83bce35006b7e035aa24791d4c0a410ba55621b48c37b5009cbcb124967fb6a8bd8975c7430354866


In [8]:
#Performance Benchmark - Speed Test

import time

algorithms = ["ML-KEM-512", "ML-KEM-768", "ML-KEM-1024", "Kyber512", "FrodoKEM-640-AES"]

print(f"{'Algorithm':<25} {'KeyGen':>10} {'Encap':>10} {'Decap':>10} {'Total':>10}")
print("-" * 65)

ITERATIONS = 1000

for algo in algorithms:
    try:
        keygen_times = []
        encap_times = []
        decap_times = []

        for _ in range(ITERATIONS):
            with oqs.KeyEncapsulation(algo) as kem:
                #KeyGen
                t0 = time.perf_counter()
                pub = kem.generate_keypair()
                keygen_times.append(time.perf_counter() - t0)

                #Encap
                t0 = time.perf_counter()
                ct,ss1 = kem.encap_secret(pub)
                encap_times.append(time.perf_counter() - t0)

                #decap
                t0 = time.perf_counter()
                ss2 = kem.decap_secret(ct)
                decap_times.append(time.perf_counter() - t0)

        avg_kg = sum(keygen_times) / ITERATIONS * 1000
        avg_en = sum(encap_times)  / ITERATIONS * 1000
        avg_de = sum(decap_times)  / ITERATIONS * 1000
        total  = avg_kg + avg_en + avg_de

        print(f"{algo:<25} {avg_kg:>9.3f}ms {avg_en:>9.3f}ms {avg_de:>9.3f}ms {total:>9.3f}ms")

    except Exception as e:
        print(f"{algo:<25}  Not available")

Algorithm                     KeyGen      Encap      Decap      Total
-----------------------------------------------------------------
ML-KEM-512                    0.309ms     0.376ms     0.463ms     1.148ms
ML-KEM-768                    0.342ms     0.390ms     0.478ms     1.210ms
ML-KEM-1024                   0.493ms     0.548ms     0.658ms     1.699ms
Kyber512                      0.177ms     0.232ms     0.250ms     0.659ms
FrodoKEM-640-AES              7.070ms     6.590ms     6.543ms    20.203ms


# Hybrid Cryptography
####  Core Idea: Use KEM to securely share a key → Use that key with AES to encrypt data → Use ML-DSA to sign & authenticate everything!

In [9]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import os

In [10]:
# ─────────────────────────────────────────────────────────────
#                       PHASE 1 — SETUP
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("            PHASE 1 — SETUP")
print("=" * 60)

# ── Alice generates ML-KEM keypair ────────────────────────────
#Alice is the RECEIVER so she owns the KEM keypair
with oqs.KeyEncapsulation("ML-KEM-768") as alice_kem_setup:
    alice_kem_public = alice_kem_setup.generate_keypair()
    alice_kem_secret = alice_kem_setup.export_secret_key()

print("\n Alice — ML-KEM Keypair Generated")
print(f"    KEM Public Key  : {alice_kem_public.hex()[:40]}...")
print(f"    KEM Secret Key  : {alice_kem_secret.hex()[:40]}...")

# ── Bob generates ML-DSA keypair ──────────────────────────────
# Bob is the SENDER, so he owns the DSA keypair for signing
with oqs.Signature("ML-DSA-65") as bob_dsa_setup:
    bob_dsa_public = bob_dsa_setup.generate_keypair()
    bob_dsa_secret = bob_dsa_setup.export_secret_key()

print("\n Bob — ML-DSA Keypair Generated")
print(f"    DSA Public Key  : {bob_dsa_public.hex()[:40]}...")
print(f"    DSA Secret Key  : {bob_dsa_secret.hex()[:40]}...")

# ── Key Exchange ───────────────────────────────────────────────
print("\n Alice  ─ KEM_pub  ──>  Bob   (so Bob can encrypt TO Alice)")
print(" Bob    ─ DSA_pub  ──>  Alice (so Alice can VERIFY Bob's messages)")
print("\n Setup Complete!\n")

            PHASE 1 — SETUP

 Alice — ML-KEM Keypair Generated
    KEM Public Key  : 876ac94f056ee1b887a8233ca8486c8ca34761e2...
    KEM Secret Key  : ed41957fea5c9c85445be8a486227f0da60dbc04...

 Bob — ML-DSA Keypair Generated
    DSA Public Key  : 16111422f39c114ee6e81ecef4ba7bd92ccf1138...
    DSA Secret Key  : 16111422f39c114ee6e81ecef4ba7bd92ccf1138...

 Alice  ─ KEM_pub  ──>  Bob   (so Bob can encrypt TO Alice)
 Bob    ─ DSA_pub  ──>  Alice (so Alice can VERIFY Bob's messages)

 Setup Complete!



In [11]:
# ─────────────────────────────────────────────────────────────
#                   PHASE 2 — BOB ENCRYPTS AND SIGNS
# ─────────────────────────────────────────────────────────────

print("=" * 60)
print("         PHASE 2 — BOB ENCRYPTS & SIGNS")
print("=" * 60)

message = b"Hello Alice! This is Quantum Safe secret message from Bob!"
print(f"\n   Original Message    : {message.decode()}")

# ── STEP 1: KEM — Encapsulate using Alice's KEM public key ────
with oqs.KeyEncapsulation("ML-KEM-768") as bob_kem:
    kem_ciphertext, shared_secret = bob_kem.encap_secret(alice_kem_public)

print(f"\n  STEP 1 — KEM Encapsulation")
print(f"   Shared Secret      : {shared_secret.hex()[:40]}...")
print(f"   KEM Ciphertext     : {kem_ciphertext.hex()[:40]}...")

# ── STEP 2: AES-256-GCM — Encrypt message with shared secret ──
aes_key = shared_secret[:32]   # 32bytes = AES256
nonce = os.urandom(12)         # 96bit random nonce
aesgcm = AESGCM(aes_key)
ciphertext = aesgcm.encrypt(nonce, message, None)

print(f"\n  STEP 2 — AES-256-GCM Encryption")
print(f"   AES Key (32B)      : {aes_key.hex()[:40]}...")
print(f"   Nonce (12B)        : {nonce.hex()}")
print(f"   AES Ciphertext     : {ciphertext.hex()[:40]}...")

# ── STEP 3: ML-DSA — Sign (kem_ct + nonce + aes_ct) ──────────
payload_to_sign = kem_ciphertext + nonce + ciphertext

with oqs.Signature("ML-DSA-65", bob_dsa_secret) as bob_signer:
    signature = bob_signer.sign(payload_to_sign)

print(f"\n  STEP 3 — ML-DSA-65 Signing")
print(f"   Payload Signed     : kem_ct + nonce + aes_ct")
print(f"   Signature          : {signature.hex()[:40]}...")

# ── Package Bob sends to Alice ─────────────────────────────────
package = {
    "kem_ciphertext": kem_ciphertext,
    "nonce"         : nonce,
    "ciphertext"    : ciphertext,
    "signature"     : signature
} 

print(f"\n  Package Dispatched to Alice!")
print(f"   Contains           : kem_ct + nonce + aes_ct + signature")
print("\n   Bob's Side Complete!\n")

         PHASE 2 — BOB ENCRYPTS & SIGNS

   Original Message    : Hello Alice! This is Quantum Safe secret message from Bob!

  STEP 1 — KEM Encapsulation
   Shared Secret      : 62933bb8437d8a4b3b5b217c45e0c3c79cd114e2...
   KEM Ciphertext     : 654be1d2dd20975633fea6b87aa832dd822604c2...

  STEP 2 — AES-256-GCM Encryption
   AES Key (32B)      : 62933bb8437d8a4b3b5b217c45e0c3c79cd114e2...
   Nonce (12B)        : 45336da404f27d8887d4907a
   AES Ciphertext     : 4dfd398511a7bf648c7904cde4ec82fce0940ad0...

  STEP 3 — ML-DSA-65 Signing
   Payload Signed     : kem_ct + nonce + aes_ct
   Signature          : a56da44c8b8afc4f652e0b4640d69ae7efe15efe...

  Package Dispatched to Alice!
   Contains           : kem_ct + nonce + aes_ct + signature

   Bob's Side Complete!



In [12]:
# ─────────────────────────────────────────────────────────────
#                PHASE 3 — ALICE VERIFIES AND DECRYPTS
# ─────────────────────────────────────────────────────────────

# ── STEP 1: ML-DSA — Verify signature using Bob's DSA public key

payload_to_verify = (
    package["kem_ciphertext"]+
    package["nonce"]+
    package["ciphertext"]
)

with oqs.Signature("ML-DSA-65") as verifier:
    is_valid = verifier.verify(
        payload_to_verify,
        package["signature"],
        bob_dsa_public                      #Bob's DSA public key
    )

print(f"\n  STEP 1 — ML-DSA-65 Signature Verification")
print(f"   Verified with      : Bob's DSA Public Key")
print(f"   Signature Valid    : {is_valid}")

if not is_valid:
    raise Exception("  Signature FAILED! Message has been tampered!")

# ── STEP 2: KEM — Decapsulate using Alice's KEM secret key ────

with oqs.KeyEncapsulation("ML-KEM-768", alice_kem_secret) as alice_recv:
    recovered_secret = alice_recv.decap_secret(package["kem_ciphertext"])
    
print(f"\n  STEP 2 — KEM Decapsulation")
print(f"   Decapsulated with  : Alice's KEM Secret Key")
print(f"   Recovered Secret   : {recovered_secret.hex()[:40]}...")

# ── Verify shared secrets match ────────────────────────────────
secrets_match = (shared_secret == recovered_secret)
print(f"   Secrets Match      : {secrets_match}")

# ── STEP 3: AES-256-GCM — Decrypt the message ─────────────────
aes_key_recvd = recovered_secret[:32]
aesgcm_recvd = AESGCM(aes_key_recvd)
decrypted = aesgcm_recvd.decrypt(
    package["nonce"],
    package["ciphertext"],
    None
)

print(f"\n  STEP 3 — AES-256-GCM Decryption")
print(f"   Decrypted Message  : {decrypted.decode()}")
print("\n  Alice's Side Complete!\n")


  STEP 1 — ML-DSA-65 Signature Verification
   Verified with      : Bob's DSA Public Key
   Signature Valid    : True

  STEP 2 — KEM Decapsulation
   Decapsulated with  : Alice's KEM Secret Key
   Recovered Secret   : 62933bb8437d8a4b3b5b217c45e0c3c79cd114e2...
   Secrets Match      : True

  STEP 3 — AES-256-GCM Decryption
   Decrypted Message  : Hello Alice! This is Quantum Safe secret message from Bob!

  Alice's Side Complete!



In [13]:
# ─────────────────────────────────────────────────────────────
#                PHASE 4 — TAMPER DETECTION TEST
# ─────────────────────────────────────────────────────────────

#Attacker modifies the ciphertext

tampered_package = dict(package)
tampered_package["ciphertext"] = b"TAMPERED!!" + package["ciphertext"]

tampered_payload = (
    tampered_package["kem_ciphertext"] +
    tampered_package["nonce"]          +
    tampered_package["ciphertext"]
)

with oqs.Signature("ML-DSA-65") as verifier:
    tampered_valid = verifier.verify(
        tampered_payload,
        tampered_package["signature"],
        bob_dsa_public
    )

print(f"\n  Tampered Signature Valid : {tampered_valid}")

if not tampered_valid:
    print("  Attack Detected! Message rejected by Alice!")
print("\n  Tamper Test Complete!\n")


  Tampered Signature Valid : False
  Attack Detected! Message rejected by Alice!

  Tamper Test Complete!



In [14]:
print("=" * 60)
print("         📊  PHASE 5 — SUMMARY REPORT")
print("=" * 60)

print(f"""
┌─────────────────────────────────────────────────────┐
│                Hybrid Encryption Summary            │
├──────────────────────────┬──────────────────────────┤
│ KEM Algorithm            │ ML-KEM-768               │
│ Signature Algorithm      │ ML-DSA-65                │
│ Symmetric Algorithm      │ AES-256-GCM              │
├──────────────────────────┼──────────────────────────┤
│ Alice owns               │ ML-KEM keypair           │
│ Bob owns                 │ ML-DSA keypair           │
├──────────────────────────┼──────────────────────────┤
│ KEM Ciphertext Size      │ {len(kem_ciphertext)} bytes               │
│ AES Ciphertext Size      │ {len(ciphertext)} bytes                 │
│ Signature Size           │ {len(signature)} bytes               │
│ Nonce Size               │ {len(nonce)} bytes                 │
├──────────────────────────┼──────────────────────────┤
│ Signature Verified       │ {is_valid}                     │
│ Secrets Match            │ {secrets_match}                     │
│ Tamper Detected          │ {not tampered_valid}                     │
│ Decryption Successful    │ {decrypted == message}                     │
└──────────────────────────┴──────────────────────────┘
""")


         📊  PHASE 5 — SUMMARY REPORT

┌─────────────────────────────────────────────────────┐
│                Hybrid Encryption Summary            │
├──────────────────────────┬──────────────────────────┤
│ KEM Algorithm            │ ML-KEM-768               │
│ Signature Algorithm      │ ML-DSA-65                │
│ Symmetric Algorithm      │ AES-256-GCM              │
├──────────────────────────┼──────────────────────────┤
│ Alice owns               │ ML-KEM keypair           │
│ Bob owns                 │ ML-DSA keypair           │
├──────────────────────────┼──────────────────────────┤
│ KEM Ciphertext Size      │ 1088 bytes               │
│ AES Ciphertext Size      │ 74 bytes                 │
│ Signature Size           │ 3309 bytes               │
│ Nonce Size               │ 12 bytes                 │
├──────────────────────────┼──────────────────────────┤
│ Signature Verified       │ True                     │
│ Secrets Match            │ True                     │
│ Tamper D

## SLH - DSA
##### Stateless hash based DSA

In [15]:
print("=" * 60)
print("         PHASE 1 — SETUP & KEY GENERATION")
print("=" * 60)

# ── SLH-DSA Algorithm Variant ─────────────────────────────────
# Available variants:
#   SPHINCS+-SHA2-128f-simple   → Fast,  128-bit security
#   SPHINCS+-SHA2-192f-simple   → Fast,  192-bit security
#   SPHINCS+-SHA2-256f-simple   → Fast,  256-bit security (FIPS 205)
#   SPHINCS+-SHA2-128s-simple   → Small, 128-bit security
#   SPHINCS+-SHA2-256s-simple   → Small, 256-bit security

ALGORITHM = "SPHINCS+-SHA2-256f-simple"  #  FIPS 205 compliant variant

#Generate SLH keypair
with oqs.Signature(ALGORITHM) as signer_setup:
    verification_key = signer_setup.generate_keypair()
    signing_key      = signer_setup.export_secret_key()

print(f"\n Algorithm            : {ALGORITHM}")
print(f" Keypair Generated!")
print(f"\n    Verification Key  : {verification_key.hex()[:40]}...")
print(f"    Signing Key       : {signing_key.hex()[:40]}...")
print(f"\n    Verification Key Size : {len(verification_key)} bytes")
print(f"    Signing Key Size      : {len(signing_key)} bytes")
print("\n Setup Complete!\n")

         PHASE 1 — SETUP & KEY GENERATION

 Algorithm            : SPHINCS+-SHA2-256f-simple
 Keypair Generated!

    Verification Key  : 52e1a644406b0854eec6fea71ee8adf8cc60ddb8...
    Signing Key       : 8fb0f91d122a347358a61df23d8f7e576a0a4155...

    Verification Key Size : 64 bytes
    Signing Key Size      : 128 bytes

 Setup Complete!



In [16]:
print("=" * 60)
print("           PHASE 2 — SIGN THE MESSAGE")
print("=" * 60)

# ── Message to Sign ────────────────────────────────────────────

message = b"Hello! This is quantum safe message signed with SLH-DSA(FIPS 205)!!!"
print(f"\n📨 Original Message     : {message.decode()}")

# ── Compute Message Hash (optional but good practice) ─────────
message_hash = hashlib.sha256(message).hexdigest()
print(f"🔍 Message SHA-256      : {message_hash[:40]}...")

# ── Sign the Message using Signing Key ────────────────────────
with oqs.Signature(ALGORITHM, signing_key) as signer:
    signature = signer.sign(message)

print(f"\n  Signature Generated!")
print(f"   Signature            : {signature.hex()[:40]}...")
print(f"   Signature Size       : {len(signature)} bytes")
print("\n Signing Complete!\n")

           PHASE 2 — SIGN THE MESSAGE

📨 Original Message     : Hello! This is quantum safe message signed with SLH-DSA(FIPS 205)!!!
🔍 Message SHA-256      : 3c2bfadb6bddd215daa7c9f133f327f518487f66...

  Signature Generated!
   Signature            : 76159d9a71b552297bd62f3cdbb05dccf5354741...
   Signature Size       : 49856 bytes

 Signing Complete!



In [17]:
print("=" * 60)
print("         PHASE 3 — VERIFY THE SIGNATURE")
print("=" * 60)

# ── Verify using Verification Key ─────────────────────────────
with oqs.Signature(ALGORITHM) as verifier:
    is_valid = verifier.verify(
        message,
        signature,
        verification_key       #Public verification key
    )

print(f"\n Verifying Signature...")
print(f"   Verified with        : Verification Key (Public)")
print(f"   Result               : {' VALID' if is_valid else ' INVALID'}")

if not is_valid:
    raise Exception(" Signature verification failed!")

         PHASE 3 — VERIFY THE SIGNATURE

 Verifying Signature...
   Verified with        : Verification Key (Public)
   Result               :  VALID


In [18]:
print("=" * 60)
print("         PHASE 4 — TAMPER DETECTION TESTS")
print("=" * 60)

# ── Test 1: Tampered Message ───────────────────────────────────
tampered_message = b"This message was TAMPERED by an attacker!"

with oqs.Signature(ALGORITHM) as verifier:
    test1 = verifier.verify(tampered_message, signature, verification_key)

print(f"\n    Test 1 — Tampered Message")
print(f"      Original  : {message.decode()[:45]}...")
print(f"      Tampered  : {tampered_message.decode()}")
print(f"      Valid     : {test1}")
print(f"      {' Rejected!' if not test1 else ' Accepted — unexpected!'}")

# ── Test 2: Forged Signature ───────────────────────────────────
forged_sig = os.urandom(len(signature))

with oqs.Signature(ALGORITHM) as verifier:
    test2 = verifier.verify(message, forged_sig, verification_key)

print(f"\n    Test 2 — Forged Signature")
print(f"      Forged Sig : {forged_sig.hex()[:40]}...")
print(f"      Valid      : {test2}")
print(f"      {' Rejected!' if not test2 else ' Accepted — unexpected!'}")

# ── Test 3: Wrong Verification Key ────────────────────────────
with oqs.Signature(ALGORITHM) as fake_setup:
    fake_vk = fake_setup.generate_keypair()

with oqs.Signature(ALGORITHM) as verifier:
    test3 = verifier.verify(message, signature, fake_vk)

print(f"\n    Test 3 — Wrong Verification Key")
print(f"      Fake VK    : {fake_vk.hex()[:40]}...")
print(f"      Valid      : {test3}")
print(f"      {' Rejected!' if not test3 else ' Accepted — unexpected!'}")

print("\n Tamper Detection Tests Complete!\n")


         PHASE 4 — TAMPER DETECTION TESTS

    Test 1 — Tampered Message
      Original  : Hello! This is quantum safe message signed wi...
      Tampered  : This message was TAMPERED by an attacker!
      Valid     : False
       Rejected!

    Test 2 — Forged Signature
      Forged Sig : 2e9c9357f3390ec6b9b4e2ebc16c94f13f21e283...
      Valid      : False
       Rejected!

    Test 3 — Wrong Verification Key
      Fake VK    : 943e4580d0fc7008f28cbf94de85ac63543d6957...
      Valid      : False
       Rejected!

 Tamper Detection Tests Complete!



In [19]:
print("=" * 60)
print("          📊  PHASE 5 — SUMMARY REPORT")
print("=" * 60)

print(f"""
┌──────────────────────────────────────────────────────────┐
│            SLH-DSA Hypertree — Summary Report            │
├─────────────────────────────┬────────────────────────────┤
│ Algorithm                   │ {ALGORITHM}  │
│ NIST Standard               │ FIPS 205                   │
│ Internal Structure          │ Hypertree (d=22 layers)    │
│ Hash Function               │ SHA-2                      │
│ Signing Mode                │ Fast (f)                   │
│ Quantum Safe                │ Yes                        │
├─────────────────────────────┼────────────────────────────┤
│ Public Size                 │ {len(verification_key)} bytes                   │
│ Private Size                │ {len(signing_key)} bytes                  │
│ Signature Size              │ {len(signature)} bytes                │
├─────────────────────────────┼────────────────────────────┤
│ Signature Valid             │ {is_valid}                       │
│ Tampered Message Rejected   │ {not test1}                       │
│ Forged Signature Rejected   │ {not test2}                       │
│ Wrong Key Rejected          │ {not test3}                       │
└─────────────────────────────┴────────────────────────────┘
""")


          📊  PHASE 5 — SUMMARY REPORT

┌──────────────────────────────────────────────────────────┐
│            SLH-DSA Hypertree — Summary Report            │
├─────────────────────────────┬────────────────────────────┤
│ Algorithm                   │ SPHINCS+-SHA2-256f-simple  │
│ NIST Standard               │ FIPS 205                   │
│ Internal Structure          │ Hypertree (d=22 layers)    │
│ Hash Function               │ SHA-2                      │
│ Signing Mode                │ Fast (f)                   │
│ Quantum Safe                │ Yes                        │
├─────────────────────────────┼────────────────────────────┤
│ Public Size                 │ 64 bytes                   │
│ Private Size                │ 128 bytes                  │
│ Signature Size              │ 49856 bytes                │
├─────────────────────────────┼────────────────────────────┤
│ Signature Valid             │ True                       │
│ Tampered Message Rejected   │ True          

In [20]:
#Digital Signature - Sign and Verify Demo

sig_algorithms = ['SPHINCS+-SHA2-128f-simple', 'SPHINCS+-SHA2-192f-simple', 'SPHINCS+-SHA2-256f-simple', 'SPHINCS+-SHA2-128s-simple', 'SPHINCS+-SHA2-256s-simple']

keygen_times = []
sign_times = []
verify_times = []
ITERATIONS = 20

message = b"Hello Post Quantum World! This message is quantum safe-signed!"

for algo in sig_algorithms:
    try:
        for _ in range(ITERATIONS):
            with oqs.Signature(algo) as signer:
                #Generate Key Pair
                t0 = time.perf_counter()
                public_key = signer.generate_keypair()
                keygen_times.append(time.perf_counter() - t0)
                
    
                #Sign
                t0 = time.perf_counter()
                signature = signer.sign(message)
                sign_times.append(time.perf_counter() - t0)
    
                #Verify
                t0 = time.perf_counter()
                is_valid = signer.verify(message, signature, public_key)
                verify_times.append(time.perf_counter() - t0)
    
        avg_kg = sum(keygen_times) / ITERATIONS * 1000
        avg_en = sum(sign_times)  / ITERATIONS * 1000
        avg_de = sum(verify_times)  / ITERATIONS * 1000
        total  = avg_kg + avg_en + avg_de

        print(f"   Algorithm   : {algo}")
        print(f"   Public Key  : {len(public_key)} bytes")
        print(f"   Signature   : {len(signature)} bytes")
        print(f"   Valid       : {'YES' if is_valid else 'NO'}")
        print()
        print(f"{'Algorithm':<25} {'KeyGen':>10} {'Signing':>10} {'Verification':>10} {'Total':>10}")
        print("-" * 65)
        print(f"{algo:<25} {avg_kg:>9.3f}ms {avg_en:>9.3f}ms {avg_de:>9.3f}ms {total:>9.3f}ms")
        print()
        print()
            
    except Exception as e:
        print(f"  {algo} not available: {e}\n")

   Algorithm   : SPHINCS+-SHA2-128f-simple
   Public Key  : 32 bytes
   Signature   : 17088 bytes
   Valid       : YES

Algorithm                     KeyGen    Signing Verification      Total
-----------------------------------------------------------------
SPHINCS+-SHA2-128f-simple     6.251ms   147.475ms     8.773ms   162.499ms


   Algorithm   : SPHINCS+-SHA2-192f-simple
   Public Key  : 48 bytes
   Signature   : 35664 bytes
   Valid       : YES

Algorithm                     KeyGen    Signing Verification      Total
-----------------------------------------------------------------
SPHINCS+-SHA2-192f-simple    15.604ms   397.420ms    22.287ms   435.312ms


   Algorithm   : SPHINCS+-SHA2-256f-simple
   Public Key  : 64 bytes
   Signature   : 49856 bytes
   Valid       : YES

Algorithm                     KeyGen    Signing Verification      Total
-----------------------------------------------------------------
SPHINCS+-SHA2-256f-simple    40.004ms   899.484ms    35.263ms   974.751ms


## Falcon

In [21]:
#Generate Falcon keys
ALGORITHM = "Falcon-1024"

with oqs.Signature(ALGORITHM) as signer_setup:
    public_key = signer_setup.generate_keypair()

    secret_key = signer_setup.export_secret_key()
    
print(f"\n Algorithm            : {ALGORITHM}")
print(f" Keypair Generated!")
print(f"\n    Verification Key  : {public_key.hex()[:40]}...")
print(f"    Signing Key       : {secret_key.hex()[:40]}...")
print(f"\n    Verification Key Size : {len(public_key)} bytes")
print(f"    Signing Key Size      : {len(secret_key)} bytes")
print("\n Setup Complete!\n")


 Algorithm            : Falcon-1024
 Keypair Generated!

    Verification Key  : 0a8e2293c4c396e92298b4b186ac7c178efd5997...
    Signing Key       : 5a008450781e005261fbc327c84fdf841f8a708f...

    Verification Key Size : 1793 bytes
    Signing Key Size      : 2305 bytes

 Setup Complete!



In [22]:
#Sign a message
message = b"I am securing this message with Falcon keys!!!"

with oqs.Signature(ALGORITHM, secret_key) as signer:
    signature = signer.sign(message)

print(f"\n  Signature Generated!")
print(f"   Signature            : {signature.hex()[:40]}...")
print(f"   Signature Size       : {len(signature)} bytes")
print("\n Signing Complete!\n")


  Signature Generated!
   Signature            : 3a42a99f9a9bfaba8c03ecd7f6f4c488e4b71eba...
   Signature Size       : 1273 bytes

 Signing Complete!



In [23]:
#Verify the message
with oqs.Signature(ALGORITHM) as verifier:
    is_valid = verifier.verify(
        message,
        signature,
        public_key
    )

print(f"\n Verifying Signature...")
print(f"   Verified with        : Verification Key (Public)")
print(f"   Result               : {' VALID' if is_valid else ' INVALID'}")

if not is_valid:
    raise Exception(" Signature verification failed!")


 Verifying Signature...
   Verified with        : Verification Key (Public)
   Result               :  VALID


In [24]:
print("=" * 60)
print("         PHASE 4 — TAMPER DETECTION TESTS")
print("=" * 60)

# ── Test 1: Tampered Message ───────────────────────────────────
tampered_message = b"This message was TAMPERED by an attacker!"

with oqs.Signature(ALGORITHM) as verifier:
    test1 = verifier.verify(tampered_message, signature, verification_key)

print(f"\n    Test 1 — Tampered Message")
print(f"      Original  : {message.decode()[:45]}...")
print(f"      Tampered  : {tampered_message.decode()}")
print(f"      Valid     : {test1}")
print(f"      {' Rejected!' if not test1 else ' Accepted — unexpected!'}")

# ── Test 2: Forged Signature ───────────────────────────────────
forged_sig = os.urandom(len(signature))

with oqs.Signature(ALGORITHM) as verifier:
    test2 = verifier.verify(message, forged_sig, verification_key)

print(f"\n    Test 2 — Forged Signature")
print(f"      Forged Sig : {forged_sig.hex()[:40]}...")
print(f"      Valid      : {test2}")
print(f"      {' Rejected!' if not test2 else ' Accepted — unexpected!'}")

# ── Test 3: Wrong Verification Key ────────────────────────────
with oqs.Signature(ALGORITHM) as fake_setup:
    fake_vk = fake_setup.generate_keypair()

with oqs.Signature(ALGORITHM) as verifier:
    test3 = verifier.verify(message, signature, fake_vk)

print(f"\n    Test 3 — Wrong Verification Key")
print(f"      Fake VK    : {fake_vk.hex()[:40]}...")
print(f"      Valid      : {test3}")
print(f"      {' Rejected!' if not test3 else ' Accepted — unexpected!'}")

print("\n Tamper Detection Tests Complete!\n")


         PHASE 4 — TAMPER DETECTION TESTS

    Test 1 — Tampered Message
      Original  : I am securing this message with Falcon keys!!...
      Tampered  : This message was TAMPERED by an attacker!
      Valid     : False
       Rejected!

    Test 2 — Forged Signature
      Forged Sig : 83a557c93260c62ec414d165f2d13ee3ac3ef0bf...
      Valid      : False
       Rejected!

    Test 3 — Wrong Verification Key
      Fake VK    : 0a17399cc78bcdd982e87379c2c935ad666937bd...
      Valid      : False
       Rejected!

 Tamper Detection Tests Complete!



In [25]:
#Digital Signature - Sign and Verify Demo

sig_algorithms = ['Falcon-512', 'Falcon-1024']

keygen_times = []
sign_times = []
verify_times = []
ITERATIONS = 1000

message = b"Hello Post Quantum World! This message is quantum safe-signed!"

for algo in sig_algorithms:
    try:
        for _ in range(ITERATIONS):
            with oqs.Signature(algo) as signer:
                #Generate Key Pair
                t0 = time.perf_counter()
                public_key = signer.generate_keypair()
                keygen_times.append(time.perf_counter() - t0)
                
    
                #Sign
                t0 = time.perf_counter()
                signature = signer.sign(message)
                sign_times.append(time.perf_counter() - t0)
    
                #Verify
                t0 = time.perf_counter()
                is_valid = signer.verify(message, signature, public_key)
                verify_times.append(time.perf_counter() - t0)
    
        avg_kg = sum(keygen_times) / ITERATIONS * 1000
        avg_en = sum(sign_times)  / ITERATIONS * 1000
        avg_de = sum(verify_times)  / ITERATIONS * 1000
        total  = avg_kg + avg_en + avg_de

        print(f"   Algorithm   : {algo}")
        print(f"   Public Key  : {len(public_key)} bytes")
        print(f"   Signature   : {len(signature)} bytes")
        print(f"   Valid       : {'YES' if is_valid else 'NO'}")
        print()
        print(f"{'Algorithm':<25} {'KeyGen':>10} {'Signing':>10} {'Verification':>10} {'Total':>10}")
        print("-" * 65)
        print(f"{algo:<25} {avg_kg:>9.3f}ms {avg_en:>9.3f}ms {avg_de:>9.3f}ms {total:>9.3f}ms")
        print()
        print()
            
    except Exception as e:
        print(f"  {algo} not available: {e}\n")

   Algorithm   : Falcon-512
   Public Key  : 897 bytes
   Signature   : 655 bytes
   Valid       : YES

Algorithm                     KeyGen    Signing Verification      Total
-----------------------------------------------------------------
Falcon-512                   31.423ms     2.328ms     0.232ms    33.983ms


   Algorithm   : Falcon-1024
   Public Key  : 1793 bytes
   Signature   : 1268 bytes
   Valid       : YES

Algorithm                     KeyGen    Signing Verification      Total
-----------------------------------------------------------------
Falcon-1024                 108.685ms     6.837ms     0.686ms   116.208ms


